## Step 1: Setup Environment and Clone Repository

In [ ]:
# Check if we're running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("ℹ️ Running in local environment")

import os
import sys
from pathlib import Path

# Set working directory
if IN_COLAB:
    os.chdir('/content')
    
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Clone the repository (replace with your actual repository URL)
REPO_URL = "https://github.com/your-username/Footaball-analysis.git"  # Update this with your repo URL
REPO_NAME = "Footaball-analysis"

# Remove existing directory if it exists
if os.path.exists(REPO_NAME):
    !rm -rf {REPO_NAME}
    print(f"Removed existing {REPO_NAME} directory")

# Clone the repository
!git clone {REPO_URL}
print(f"✅ Repository cloned successfully")

# Change to repository directory
os.chdir(REPO_NAME)
print(f"Changed to directory: {os.getcwd()}")

# List directory contents
!ls -la

## Step 2: Install Dependencies

In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-glx libglib2.0-0 libsm6 libxext6 libxrender-dev libgomp1 libgoogle-perftools4

# Install Python packages
!pip install -q supervision==0.27.0
!pip install -q ultralytics>=8.3.235
!pip install -q transformers>=4.57.3
!pip install -q tokenizers==0.22.1
!pip install -q autoprocessor>=0.9.0
!pip install -q gdown>=5.2.0
!pip install -q roboflow>=1.2.11
!pip install -q python-dotenv>=1.2.1
!pip install -q tqdm>=4.67.1

print("✅ Dependencies installed successfully")

## Step 3: Setup Python Path and Imports

In [ ]:
# Add the project to Python path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path: {sys.path[:3]}...")  # Show first few entries

# Test imports
try:
    import supervision as sv
    from ultralytics import YOLO
    import torch
    from datetime import datetime
    from typing import Tuple
    
    print("✅ Basic imports successful")
    print(f"Supervision version: {sv.__version__}")
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
        
except ImportError as e:
    print(f"❌ Import error: {e}")

In [ ]:
# Import project modules
try:
    from src.model import get_models
    from src.team import TeamClassifier, TeamConsistencyTracker
    from src.frame_processor import initialize_frame_processor, process_frame
    from src.annotators import get_annotators, get_tracker
    from src.config import (
        OUTPUT_DIR,
        PLAYER_DETECTION_MODEL_PATH,
        KEYPOINT_MODEL_PATH,
        BALL_ID,
        GOALKEEPER_ID,
        PLAYER_ID,
        REFEREE_ID,
    )
    
    print("✅ Project modules imported successfully")
    
except ImportError as e:
    print(f"❌ Project import error: {e}")
    print("Make sure the repository was cloned correctly and contains all source files")

## Step 4: Upload Video File and Check Models

In [ ]:
# Upload video file
if IN_COLAB:
    from google.colab import files
    
    print("Please upload your video file:")
    uploaded = files.upload()
    
    # Get the uploaded file name
    video_filename = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {video_filename}")
    
    # Move to inputs directory
    inputs_dir = Path("inputs")
    inputs_dir.mkdir(exist_ok=True)
    
    import shutil
    video_path = inputs_dir / video_filename
    shutil.move(video_filename, video_path)
    
    print(f"✅ Video moved to: {video_path}")
else:
    # For local environment, specify your video path
    video_path = input("Please enter the path to your video file: ")
    video_path = Path(video_path)
    
if not video_path.exists():
    print(f"❌ Video file not found: {video_path}")
else:
    print(f"✅ Video file ready: {video_path}")
    INPUT_VIDEO_PATH = str(video_path)

In [ ]:
# Check if model files exist
models_dir = Path("models")
player_model_path = models_dir / "player-detection.pt"
keypoint_model_path = models_dir / "keypoint-detection.pt"

print("Checking model files:")
print(f"Models directory: {models_dir.exists()}")
print(f"Player detection model: {player_model_path.exists()}")
print(f"Keypoint detection model: {keypoint_model_path.exists()}")

if not player_model_path.exists() or not keypoint_model_path.exists():
    print("\n⚠️ Model files not found. You may need to:")
    print("1. Download the model files manually")
    print("2. Or use pre-trained YOLO models for testing")
    
    # Option to use default YOLO models
    use_default = input("\nUse default YOLO models for testing? (y/n): ").lower() == 'y'
    
    if use_default:
        print("Using default YOLO models...")
        # Will be handled in the model loading section
        USE_DEFAULT_MODELS = True
    else:
        print("Please upload your model files to the models/ directory")
        USE_DEFAULT_MODELS = False
else:
    print("✅ All model files found")
    USE_DEFAULT_MODELS = False

## Step 5: Initialize Models and Pipeline Components

In [ ]:
def create_output_directory() -> Path:
    """Create output directory with timestamp"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = Path(OUTPUT_DIR) / f"analysis_{timestamp}"
    output_path.mkdir(parents=True, exist_ok=True)
    return output_path

# Create output directory
output_dir = create_output_directory()
print(f"✅ Output directory created: {output_dir}")

In [ ]:
def setup_models_and_annotators() -> Tuple:
    """Initialize all models and annotators"""
    print("Loading models...")
    
    if USE_DEFAULT_MODELS:
        # Use default YOLO models for testing
        print("Using default YOLO models...")
        player_detection_model = YOLO("yolov8n.pt")
        keypoint_model = YOLO("yolov8n-pose.pt")
    else:
        # Load custom models
        player_detection_model, keypoint_model = get_models(
            player_detection_model_path=PLAYER_DETECTION_MODEL_PATH,
            keypoint_model_path=KEYPOINT_MODEL_PATH,
        )
    
    # Initialize team classifier
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    team_classifier = TeamClassifier(device=device)
    
    # Initialize tracker
    tracker = get_tracker()
    tracker.reset()
    
    # Initialize team tracker
    team_tracker = TeamConsistencyTracker()
    
    # Setup annotators
    ellipse_annotator, label_annotator, triangle_annotator = get_annotators()
    
    print("✅ Models and annotators loaded successfully!")
    
    return (
        player_detection_model,
        keypoint_model,
        team_classifier,
        tracker,
        team_tracker,
        ellipse_annotator,
        label_annotator,
        triangle_annotator,
    )

# Load models and annotators
try:
    (
        player_detection_model,
        keypoint_model,
        team_classifier,
        tracker,
        team_tracker,
        ellipse_annotator,
        label_annotator,
        triangle_annotator,
    ) = setup_models_and_annotators()
    
except Exception as e:
    print(f"❌ Error loading models: {e}")
    raise

## Step 6: Process the Video

In [ ]:
# Initialize frame processor
print("Initializing frame processor...")

try:
    initialize_frame_processor(
        player_detection_model=player_detection_model,
        keypoint_model=keypoint_model,
        tracker_obj=tracker,
        team_classifier_obj=team_classifier,
        team_tracker_obj=team_tracker,
        ellipse_ann=ellipse_annotator,
        label_ann=label_annotator,
        triangle_ann=triangle_annotator,
        ball_id=BALL_ID,
        goalkeeper_id=GOALKEEPER_ID,
        player_id=PLAYER_ID,
        referee_id=REFEREE_ID,
    )
    
    print("✅ Frame processor initialized successfully!")
    
except Exception as e:
    print(f"❌ Error initializing frame processor: {e}")
    raise

In [ ]:
def process_video(input_path: str, output_path: Path, max_frames: int = None) -> Path:
    """Process the entire video and save output"""
    
    print(f"Processing video: {input_path}")
    
    # Get video info
    video_info = sv.VideoInfo.from_video_path(input_path)
    print(
        f"Video info: {video_info.width}x{video_info.height}, {video_info.fps}fps, {video_info.total_frames} frames"
    )
    
    # Limit frames for testing in Colab
    if max_frames and video_info.total_frames > max_frames:
        print(f"⚠️ Limiting processing to {max_frames} frames for Colab performance")
        total_frames_to_process = max_frames
    else:
        total_frames_to_process = video_info.total_frames
    
    # Create output video path
    output_video_path = output_path / f"analyzed_{Path(input_path).name}"
    
    print(f"Output video will be saved to: {output_video_path}")
    
    # Process video
    with sv.VideoSink(str(output_video_path), video_info) as sink:
        frame_generator = sv.get_video_frames_generator(input_path)
        
        for frame_idx, frame in enumerate(frame_generator):
            if max_frames and frame_idx >= max_frames:
                break
                
            # Process frame
            processed_frame = process_frame(frame, frame_idx)
            
            # Write processed frame
            sink.write_frame(processed_frame)
            
            # Progress update
            if frame_idx % 30 == 0:  # Every second at 30fps
                progress = (frame_idx + 1) / total_frames_to_process * 100
                print(
                    f"Progress: {progress:.1f}% ({frame_idx + 1}/{total_frames_to_process} frames)",
                    end="\r"
                )
    
    print(f"\n✅ Video processing completed! Output saved to: {output_video_path}")
    return output_video_path

# Set maximum frames for Colab (to prevent timeout)
# Remove this limit or increase it based on your needs
MAX_FRAMES = 300  # Process first 10 seconds at 30fps

print(f"=== Starting Video Processing ===")
print(f"Input video: {INPUT_VIDEO_PATH}")
print(f"Output directory: {output_dir}")

try:
    # Process the video
    output_video_path = process_video(INPUT_VIDEO_PATH, output_dir, MAX_FRAMES)
    
    print("\n=== Processing Complete ===")
    print(f"📹 Processed video: {output_video_path}")
    print(f"📁 Output directory: {output_dir}")
    print("\nFeatures applied:")
    print("  ✅ Player detection and tracking")
    print("  ✅ Ball detection")
    print("  ✅ Team classification")
    print("  ✅ Tactical mini-map overlay")
    print("  ✅ Real-time annotations")
    
except Exception as e:
    print(f"❌ Error during processing: {e}")
    import traceback
    traceback.print_exc()

## Step 7: Download Results

In [ ]:
# Download the processed video
if IN_COLAB:
    from google.colab import files
    
    try:
        print("Preparing download...")
        files.download(str(output_video_path))
        print("✅ Download initiated! Check your downloads folder.")
    except Exception as e:
        print(f"❌ Download error: {e}")
        print(f"You can manually download the file from: {output_video_path}")
else:
    print(f"✅ Results available at: {output_video_path}")
    print(f"📁 Full output directory: {output_dir}")

## Step 8: Display Video Info and Statistics

In [ ]:
# Display processing statistics
import os

if output_video_path.exists():
    # Get file sizes
    input_size = os.path.getsize(INPUT_VIDEO_PATH) / (1024 * 1024)  # MB
    output_size = os.path.getsize(output_video_path) / (1024 * 1024)  # MB
    
    print("=== Video Processing Statistics ===")
    print(f"📁 Input file size: {input_size:.1f} MB")
    print(f"📁 Output file size: {output_size:.1f} MB")
    print(f"📊 Size change: {((output_size - input_size) / input_size * 100):+.1f}%")
    
    # List all files in output directory
    print(f"\n📂 Output directory contents:")
    for file in output_dir.iterdir():
        if file.is_file():
            size = os.path.getsize(file) / (1024 * 1024)
            print(f"  📄 {file.name} ({size:.1f} MB)")
else:
    print("❌ Output video file not found")

## Additional Configuration Options

You can modify the following variables in the cells above to customize the analysis:

- `MAX_FRAMES`: Limit the number of frames to process (useful for testing)
- `USE_DEFAULT_MODELS`: Set to `True` to use default YOLO models instead of custom ones
- Model parameters in the configuration

### Troubleshooting

1. **Out of memory**: Reduce `MAX_FRAMES` or use CPU instead of GPU
2. **Model not found**: Make sure model files are in the `models/` directory
3. **Import errors**: Ensure all dependencies are installed correctly
4. **Video upload issues**: Make sure the video file is in a supported format (mp4, avi, etc.)

### Performance Tips

1. Use GPU runtime in Colab for better performance
2. Process shorter videos or limit frames for testing
3. Consider reducing video resolution if processing is slow
4. Monitor memory usage in Colab to avoid crashes

---

**Happy analyzing! 🏈⚽**